# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one content item (page) belonging to one client (`content_hash_id` + `client_hash_id`).

**Time window:** We will test on a mid-panel month: **March 2026** (`month=2026-03`). Our features will be calculated from the prior 30 days (**February 2026**), and our target label will be the page's performance in March 2026. This enforces a strict forward-window time series split.

**Tables Used:** `fact_content_daily_performance` (for daily metrics) and `dim_content` (for page age and word count).

## 2. Fields: feature / label / context / excluded

- **Feature (knowable BEFORE March 1st):**
  - `imp_feb` (available because we calculate it strictly from Feb data)
  - `clk_feb` (available because we calculate it strictly from Feb data)
  - `pos_feb` (available because we calculate it strictly from Feb data)
  - `content_age_days` (available because we know publish dates immediately)
  - `word_count` (available because content size is fixed/known before March)

- **Label / Proxy:** `is_declining_in_march` (Binary: 1 if `imp_march` < 0.8 * `imp_feb`, else 0). Never used as a feature.

- **Context:** `client_hash_id`, `content_hash_id`, `report_date`. Used only for grouping/joining, never for the model to learn from.

- **Excluded:** `ga4_sessions`. Why? Not all clients have Google Analytics 4 connected (`ga4_data_available` is False for many). Using it injects massive non-random missingness (zeros) that the model will misinterpret as "zero traffic."

## 3. Verify it with queries (grain, counts, missing values, windows)

Here we verify our grain, dates, and GA4 availability on the March partition, build our 5 features, and deliberately trip the Leakage Trap to demonstrate why we must separate our feature window from our target window.

In [2]:
import duckdb
import os
import getpass
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Authenticate
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# ---------------------------------------------------------
# QUERY 1: Grain verification (dim_content must be 1 row per content_hash_id)
# ---------------------------------------------------------
grain_check = con.sql(f"""
    SELECT content_hash_id, COUNT(*) c
    FROM read_parquet('{REL}/dim_content.parquet')
    GROUP BY content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print(f"[Query 1] Grain violations in dim_content (should be 0): {len(grain_check)}\n")

# ---------------------------------------------------------
# QUERY 2: Row count & date span for the target month (March 2026)
# ---------------------------------------------------------
dates_check = con.sql(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as start_date, MAX(report_date) as end_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("[Query 2] March 2026 Fact Table Span:")
print(dates_check, "\n")

# ---------------------------------------------------------
# QUERY 3: Availability Filter (GA4 Data missingness check)
# ---------------------------------------------------------
ga4_check = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_ga4
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("[Query 3] GA4 Availability (Why we excluded it):")
print(ga4_check)
survival_rate = (ga4_check['rows_with_ga4'][0] / ga4_check['total_rows'][0]) * 100
print(f"Survival Rate: {survival_rate:.1f}% (Too many zeros to trust!)\n")

# ---------------------------------------------------------
# FEATURE BUILDING & THE LEAKAGE TRAP
# ---------------------------------------------------------
# Build the 5 features from Feb, and the Target from March
print("Downloading grouped features over the network...")
features = con.sql(f"""
    WITH march_data AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_march
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY 1
    ),
    feb_data AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_feb,
               SUM(gsc_clicks) AS clk_feb,
               AVG(gsc_avg_position) AS pos_feb
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
        GROUP BY 1
    )
    SELECT f.content_hash_id,
           f.imp_feb, f.clk_feb, f.pos_feb, -- Features
           m.imp_march,                     -- Used for Label
           date_diff('day', d.content_created_date, DATE '2026-03-01') AS content_age_days, -- Converted Date to Age Feature
           d.word_count                     -- Features
    FROM feb_data f
    JOIN march_data m ON f.content_hash_id = m.content_hash_id
    JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
    WHERE f.imp_feb >= 100 -- Noise filter
""").df()

# Define the label (Did it decline in March?)
features['is_declining_in_march'] = (features['imp_march'] < 0.8 * features['imp_feb']).astype(int)

# THE TRAP: Intentionally cheating by including `imp_march` (the future!) as a feature
print("--- THE LEAKAGE TRAP ---")
trap_cols = ['imp_feb', 'clk_feb', 'pos_feb', 'content_age_days', 'word_count', 'imp_march'] # imp_march is leaking the answer!
X_trap = features.dropna(subset=trap_cols)[trap_cols]
y_trap = features.dropna(subset=trap_cols)['is_declining_in_march']

X_tr, X_te, y_tr, y_te = train_test_split(X_trap, y_trap, test_size=0.25, random_state=42)
trap_model = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr, y_tr)
print(f"Accuracy WITH leaked future feature (imp_march): {trap_model.score(X_te, y_te):.3f} (Suspiciously perfect!)")

# THE HONEST SCORE: Deleting the future data and relying only on the 5 Feb features
print("\n--- THE HONEST MODEL ---")
honest_cols = ['imp_feb', 'clk_feb', 'pos_feb', 'content_age_days', 'word_count']
X_honest = features.dropna(subset=honest_cols)[honest_cols]
y_honest = features.dropna(subset=honest_cols)['is_declining_in_march']

X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_honest, y_honest, test_size=0.25, random_state=42)
honest_model = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr_h, y_tr_h)
print(f"Accuracy WITHOUT leaked feature: {honest_model.score(X_te_h, y_te_h):.3f}")

[Query 1] Grain violations in dim_content (should be 0): 0

[Query 2] March 2026 Fact Table Span:
   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31 

[Query 3] GA4 Availability (Why we excluded it):
   total_rows  rows_with_ga4
0     9841378       413966.0
Survival Rate: 4.2% (Too many zeros to trust!)

--- THE LEAKAGE TRAP ---
Accuracy WITH leaked future feature (imp_march): 0.980 (Suspiciously perfect!)

--- THE HONEST MODEL ---
Accuracy WITHOUT leaked feature: 0.857


## 4. Data limits

**Limitation of this slice:** The history depth differs wildly per client (as warned in the `flyrank-data` skill). Some clients only started tracking GSC data in mid-2025 or 2026. If we try to enforce a rigid 90-day feature lookback window globally across the entire dataset, we will silently drop clients who only have 30 days of history, creating a survivorship bias where our model only learns from the oldest clients.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.